# 06 — Vasicek Model: Calibration from FRED Treasury Data

Spec §7.1–7.3. Data source: FRED series `GS1` (1-Year Treasury Constant Maturity Rate), 2000-present.

The Vasicek model: `dr_t = κ(θ - r_t)dt + σdW_t`

Calibration via OLS on the AR(1) discretisation:
```
r_{t+1} = a + b·r_t + ε   →   κ = -ln(b)/dt,  θ = a/(1-b),  σ from residual std
```

Pipeline:
1. Fetch GS1 from FRED (with CSV cache)
2. Plot the rate history
3. Calibrate κ, θ, σ via OLS
4. Verify long-run distribution (mean = θ, std = σ/√(2κ))
5. Closed-form yield curve at calibrated parameters + current rate

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import vasicek as v

FRED_DIR = Path('..') / 'data' / 'fred'
FIGURES  = Path('..') / 'artifacts' / 'figures'
FRED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Fetch GS1 from FRED

In [ ]:
kappa, theta, sigma, initial_rate = v.load_and_calibrate(
    start='2000-01-01',
    end='2025-12-31',
    cache_path=FRED_DIR / 'gs1_monthly.csv',
)
print(f'\nCalibrated parameters:')
print(f'  κ (mean-reversion speed):  {kappa:.4f}  (half-life = {np.log(2)/kappa*12:.1f} months)')
print(f'  θ (long-run mean):         {theta:.4f}  ({theta*100:.2f}%)')
print(f'  σ (volatility):            {sigma:.6f} ({sigma*100:.4f}% per √year)')
print(f'  r₀ (current short rate):  {initial_rate:.4f}  ({initial_rate*100:.2f}%)')

## 2. Plot historical GS1 rate

In [ ]:
cache_path = FRED_DIR / 'gs1_monthly.csv'
gs1 = pd.read_csv(cache_path, index_col=0, parse_dates=True)
rate_series = gs1.iloc[:, 0] / 100.0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: full history
axes[0].plot(rate_series.index, rate_series.values * 100, linewidth=1, color='steelblue')
axes[0].axhline(theta * 100, color='red', linestyle='--', linewidth=1.5,
                label=f'Long-run mean θ = {theta*100:.2f}%')
axes[0].set_title('1-Year Treasury Rate (GS1), 2000–2025')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Rate (%)')
axes[0].legend()

# Right: first-differenced rates (innovations)
diffs = rate_series.diff().dropna()
axes[1].plot(diffs.index, diffs.values * 100, linewidth=0.8, alpha=0.7, color='gray')
axes[1].axhline(0, color='k', linewidth=0.8)
axes[1].set_title('Monthly Rate Changes (innovations)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Δ Rate (%)')

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_gs1_history.png', bbox_inches='tight')
plt.show()
print(f'Rate range: {rate_series.min()*100:.2f}% – {rate_series.max()*100:.2f}%')

## 3. Verify long-run distribution (spec §7.3)

Under the Vasicek model, the stationary distribution is:
- Mean = θ
- Std  = σ / √(2κ)
- Variance = σ² / (2κ)

In [ ]:
lr_mean, lr_std, lr_var = v.long_run_distribution(kappa, theta, sigma)
print('Long-run stationary distribution:')
print(f'  Theoretical mean:  {lr_mean*100:.2f}%  (= θ)')
print(f'  Theoretical std:   {lr_std*100:.2f}%  (= σ/√(2κ))')
print(f'  Theoretical var:   {lr_var:.6f}')

# Compare to empirical moments of GS1
emp_mean = rate_series.mean()
emp_std  = rate_series.std()
print(f'\nEmpirical GS1 moments (2000-2025):')
print(f'  Empirical mean: {emp_mean*100:.2f}%')
print(f'  Empirical std:  {emp_std*100:.2f}%')
print(f'\nTheoretical/Empirical mean ratio: {lr_mean/emp_mean:.3f}')
print(f'Theoretical/Empirical std ratio:  {lr_std/emp_std:.3f}')

# Sanity: stationary std should be within 2x of empirical
assert lr_std / emp_std < 5.0, f'Stationary std implausibly far from empirical: {lr_std/emp_std:.2f}x'

## 4. Closed-form yield curve (spec §7.2)

P(t,T) = A(t,T) · exp(-B(t,T) · r_t)  
Yield(τ) = -ln(P) / τ

In [ ]:
maturities = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 15, 20, 30]
yc = v.vasicek_yield_curve(initial_rate, kappa, theta, sigma, maturities=maturities)
print('Vasicek yield curve at current short rate r₀ = {:.2f}%:'.format(initial_rate*100))
print(yc.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Plot the yield curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(yc['maturity_yrs'], yc['yield_pct'], 'o-', color='steelblue',
        linewidth=2, markersize=6, label=f'Vasicek (r₀={initial_rate*100:.1f}%)')
ax.axhline(theta * 100, color='red', linestyle='--', alpha=0.7,
           label=f'Long-run mean θ = {theta*100:.2f}%')
ax.set_title('Vasicek Model: Zero-Coupon Yield Curve')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_yield_curve.png', bbox_inches='tight')
plt.show()

## 5. Yield curve sensitivity to r₀

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for r0_scenario, color, label in [
    (0.01, 'green',     'r₀ = 1% (low rate)'),
    (0.03, 'steelblue', 'r₀ = 3%'),
    (initial_rate, 'black', f'r₀ = {initial_rate*100:.1f}% (current)'),
    (0.06, 'orange',    'r₀ = 6%'),
    (0.08, 'red',       'r₀ = 8% (high rate)'),
]:
    yc_s = v.vasicek_yield_curve(r0_scenario, kappa, theta, sigma, maturities=maturities)
    ax.plot(yc_s['maturity_yrs'], yc_s['yield_pct'], 'o-',
            color=color, linewidth=1.5, markersize=4, label=label)

ax.axhline(theta * 100, color='gray', linestyle=':', alpha=0.8,
           label=f'θ = {theta*100:.2f}%')
ax.set_title('Vasicek Yield Curves at Different r₀ Values')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('Yield (%)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'vasicek_yield_curve_scenarios.png', bbox_inches='tight')
plt.show()

print('Note: all curves converge to θ at long maturities (mean-reversion)')